# AF 주 모델: MIMIC 오탐 개선
**최종본 확정 보류.** MIMIC 시험 비AF 303개 중 91개 오탐을 개선합니다. 품질 AI 통합은 하지 않습니다.
1. T4·Drive 연결 → 2. 검증군에서 판정 기준 선택 → 3. 고정 후 기존 시험군 비교.
검증군의 두 자료 각각 특이도90%를 요구하고, 낮은 쪽 민감도를 최대화합니다. 80/85/95% 조건은 비교표로만 제시합니다. 시험군을 보고 가장 좋아 보이는 조건을 고르지 않습니다.
기존 시험 결과가 개선 방향에 영향을 줬으므로 재사용 시험 성능은 독립 검증이 아닙니다. 민감도도 같이 확인합니다. 원본 재다운로드·재학습 없이 저장된 모델을 먼저 진단합니다.

In [ ]:
from pathlib import Path
import subprocess,sys,torch
assert torch.cuda.is_available(),'런타임 유형을 T4 GPU로 설정하세요'
from google.colab import drive
drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/Capdyssey/af_integrated')
assert (base/'models/main_two_sources_v1/candidate/model.pt').exists(),'Drive 저장 모델을 확인하세요'
repo=Path('/content/capdyssey_fp_audit')
if not repo.exists():subprocess.run(['git','clone','https://github.com/dung-ho03/capdyssey.git',str(repo)],check=True)
subprocess.run(['git','fetch','origin','main'],cwd=repo,check=True)
subprocess.run(['git','checkout','83cfe88'],cwd=repo,check=True)
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
with (base/'false_positive_audit_v1.log').open('a') as log:
    with subprocess.Popen([sys.executable,'-u',str(repo/'scripts/audit_af_false_positives.py'),'--base',str(base)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as p:
        for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        assert p.wait()==0,'분석 오류: 최종본으로 승격하지 않습니다'

In [ ]:
import json
from IPython.display import HTML,display
report=json.loads((base/'models/false_positive_audit_v1/report.json').read_text())
names={'mimic_perform_af':'MIMIC','six_rhythms':'6종 PPG'}
html='<h3>오탐 개선 비교 — 기존 시험군 개발 평가</h3><table border=1 cellpadding=8><tr><th>자료</th><th>기준</th><th>민감도</th><th>특이도</th><th>오탐 수</th></tr>'
for key,label in [('previous_by_source','이전'),('new_by_source','수정 후보')]:
    for s,v in report[key].items():html+=f'<tr><td>{names[s]}</td><td>{label}</td><td>{v["sensitivity"]:.1%}</td><td>{v["specificity"]:.1%}</td><td>{v["fp"]}</td></tr>'
display(HTML(html+'</table>'))
print('선택:',report['selection']['choice']['seed'],report['selection']['choice']['threshold'])
print('두 자료 모두 민감도/특이도90% 달성:',report['goal_met'])
print('결과가 부족하면 후보로 유지합니다. 자동으로 최종본을 교체하지 않습니다.')